# Phase 6 — Baseline Machine Learning Models
### Problem Statement ID 26184: Predictive Analytics Framework for Cybercrime Complaints

## 1. Phase 6 Objective
Establish standardized machine-learning baseline benchmarks using DummyClassifier, Logistic Regression, and Random Forest models. Preprocessing is strictly fit only on `X_train`, and models are evaluated on the out-of-time `X_validation` cohort without using the holdout `X_test` dataset.


## 2. Load Train / Validation / Test Datasets
Loading partitioned datasets created in Phase 5.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROC_DIR = Path('../data/processed')
train_df = pd.read_csv(PROC_DIR / 'train.csv')
val_df = pd.read_csv(PROC_DIR / 'validation.csv')
test_df = pd.read_csv(PROC_DIR / 'test.csv')

print(f'Train      : {train_df.shape[0]:,} rows x {train_df.shape[1]} cols')
print(f'Validation : {val_df.shape[0]:,} rows x {val_df.shape[1]} cols')
print(f'Test       : {test_df.shape[0]:,} rows x {test_df.shape[1]} cols')


## 3. Target Distribution
Verifying class prevalence across temporal cohorts.


In [ ]:
target_dist = pd.read_csv('../outputs/phase6_target_distribution.csv')
target_dist


## 4. Feature Selection
Quarantining target, outcome indicators, unique IDs, and PII tokens.


In [ ]:
feat_sel = pd.read_csv('../outputs/phase6_selected_features.csv')
safe_features = feat_sel[feat_sel['selected']]['feature'].tolist()
print(f'Selected Safe Predictive Features: {len(safe_features)}')
X_train = train_df[safe_features]
y_train = train_df['future_withdrawal'].values
X_val = val_df[safe_features]
y_val = val_df['future_withdrawal'].values


## 5. Feature Type Detection
Separating features into 52 numerical and 12 categorical predictors.


In [ ]:
type_rep = pd.read_csv('../outputs/phase6_feature_type_report.csv')
print(f'Feature Types Summary:')
print(type_rep['feature_type'].value_counts())


## 6. Training-Only Preprocessing Pipelines
Configuring ColumnTransformers fit strictly on training data.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Scaled pipeline for linear models
pre_scaled = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

# Unscaled pipeline for tree-based models
pre_unscaled = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])


## 7. Baseline Model 1: DummyClassifier
Establishing naive majority-class benchmark.


In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='prior')
dummy.fit(X_train, y_train)
proba_dummy = dummy.predict_proba(X_val)[:, 1]
pred_dummy = dummy.predict(X_val)
print('Dummy Classifier fitted.')


## 8. Baseline Model 2: Logistic Regression
Fitting class-weighted linear model.


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_pipe = Pipeline([('pre', pre_scaled), ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))])
lr_pipe.fit(X_train, y_train)
proba_lr = lr_pipe.predict_proba(X_val)[:, 1]
pred_lr = (proba_lr >= 0.5).astype(int)
print('Logistic Regression fitted.')


## 9. Baseline Model 3: Random Forest
Fitting class-weighted tree ensemble.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipe = Pipeline([('pre', pre_unscaled), ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced'))])
rf_pipe.fit(X_train, y_train)
proba_rf = rf_pipe.predict_proba(X_val)[:, 1]
pred_rf = (proba_rf >= 0.5).astype(int)
print('Random Forest fitted.')


## 10. Validation Metrics
Evaluating Accuracy, Precision, Recall, F1, ROC-AUC, and PR-AUC.


In [ ]:
metrics_df = pd.read_csv('../outputs/phase6_validation_metrics.csv')
metrics_df


## 11. Confusion Matrices
Visualizing class-level prediction distributions.


In [ ]:
from IPython.display import Image, display
display(Image(filename='../outputs/figures/confusion_matrix_dummy_prior.png'))
display(Image(filename='../outputs/figures/confusion_matrix_logistic_regression.png'))
display(Image(filename='../outputs/figures/confusion_matrix_random_forest.png'))


## 12. ROC Curve Comparison
Comparing True Positive Rate vs False Positive Rate.


In [ ]:
Image(filename='../outputs/figures/validation_roc_comparison.png')


## 13. Precision-Recall Curve Comparison
Evaluating precision across recall levels on imbalanced target.


In [ ]:
Image(filename='../outputs/figures/validation_pr_comparison.png')


## 14. Decision Threshold Analysis
Evaluating operational precision-recall trade-offs.


In [ ]:
th_df = pd.read_csv('../outputs/phase6_threshold_analysis.csv')
th_df


## 15. Feature Importance & Coefficients
Analyzing top ranking drivers in Random Forest and linear weights in Logistic Regression.


In [ ]:
rf_fi = pd.read_csv('../outputs/random_forest_feature_importance.csv')
print('Top 10 Random Forest Features:')
display(rf_fi.head(10))
lr_coef = pd.read_csv('../outputs/logistic_regression_coefficients.csv')
print('Top 10 Logistic Regression Coefficients:')
display(lr_coef.head(10))


## 16. Leakage Audit
Verifying strict absence of target or test set contamination.


In [ ]:
leakage_df = pd.read_csv('../outputs/phase6_leakage_audit.csv')
leakage_df


## 17. Model Comparison
Selecting best performing baseline for future benchmarking.


In [ ]:
comp_df = pd.read_csv('../outputs/phase6_model_comparison.csv')
comp_df


## 18. Conclusion & Phase 7 Readiness
- **Best Baseline:** `logistic_regression` provides the strongest initial balanced recall (38.19%) and PR-AUC (0.1060).
- `random_forest` achieves strong ranking discrimination with up to 93.75% recall under decision threshold tuning.
- **The pipeline is 100% verified and READY for Phase 7 — XGBoost Model Development.**
